# Demo hỗ trợ cố vấn học tập: từ câu hỏi đến hành động có người duyệt

**Nhóm N02 · Khoa học Phân tích Dữ liệu**

Notebook này là phần demo của dự án, đi đúng kịch bản bảy bước trong báo cáo Milestone 5 (Mục 11.8):

| Bước | Nội dung | Ô trong notebook |
|---|---|---|
| 1 | Câu hỏi của cố vấn | Bước 1 |
| 2 | Đầu vào: dữ liệu gốc, kiểm tra SHA-256, chuẩn bị dữ liệu | Bước 2 |
| 3 | Xử lý: mô hình M1 theo đúng cấu hình đã đăng ký trước | Bước 3 |
| 4 | Đầu ra: danh sách ưu tiên kèm reason code | Bước 4 |
| 5 | Bằng chứng: precision, recall, AUC, công bằng theo nhóm | Bước 5 |
| 6 | Con người phê duyệt từng ca (cổng G2) và nhật ký kiểm toán | Bước 6 |
| 7 | Hành động: xuất danh sách đã duyệt, từ chối nếu còn ca chưa duyệt | Bước 7 |

**Cách chạy:** mở trên Google Colab, chọn *Runtime → Run all* (khoảng 1–2 phút). Dữ liệu gốc đã nhúng sẵn trong notebook, không cần tải thêm.

**Nguồn mã:** phần chuẩn bị dữ liệu và mô hình được chép nguyên văn từ notebook Milestone 3 và Milestone 4 của nhóm. Notebook này chỉ thêm reason code, danh sách cho cố vấn, cổng phê duyệt và nhật ký.

**Giới hạn cần nói rõ khi demo:**
- Dữ liệu là bộ khảo sát mở SAPE của một trường ở Bangladesh, không phải sinh viên FIT-HCMUS.
- Công cụ chỉ là lớp xếp ưu tiên bổ trợ (AUC khoảng 0,62), không phải dự báo; điểm số không đọc như xác suất.
- Danh sách chỉ dùng mã giả danh; giới tính và nơi ở không xuất hiện trong danh sách, chỉ dùng để kiểm toán công bằng ở dạng số gộp.
- Mọi quyết định liên hệ do cố vấn đưa ra; notebook không gửi bất kỳ thông báo nào.

## Bước 1. Câu hỏi của cố vấn

In [ ]:
#@title Câu hỏi của cố vấn (có thể sửa rồi chạy lại)
CAPACITY = 153  #@param {type:"integer"}
ADVISOR = "Cố vấn A (demo)"  #@param {type:"string"}
print(f"{ADVISOR}: \"Học kỳ này tôi chỉ gọi được khoảng {CAPACITY} sinh viên, nên gọi ai trước, và vì sao?\"")

## Bước 2. Đầu vào: dữ liệu gốc và chuẩn bị dữ liệu

Các ô dưới đây chép nguyên văn Phần A của notebook Milestone 4 (mã chuẩn bị dữ liệu của Milestone 3): ghi tệp gốc nhúng sẵn, kiểm tra SHA-256 và schema, làm sạch, tạo nhãn và mẫu chính 1.015 sinh viên.

In [ ]:
import hashlib
import json
import platform
import re
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import Markdown, display
from matplotlib.ticker import FuncFormatter
from scipy import stats
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 90)

BASE = Path("/content") if Path("/content").exists() else Path.cwd()
ROOT = BASE / "N02_Demo"
raw_path = ROOT / "data" / "raw" / "Students_Performance_data_set.xlsx"
interim = ROOT / "data" / "interim"
processed = ROOT / "data" / "processed"
meta = ROOT / "data" / "metadata"
FIG = ROOT / "reports" / "milestone3" / "figures"
TAB = ROOT / "reports" / "milestone3" / "tables"
for d in (raw_path.parent, interim, processed, meta, FIG, TAB):
    d.mkdir(parents=True, exist_ok=True)

print("Python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__,
      "| scipy", scipy.__version__, "| scikit-learn", sklearn.__version__)
print("Thư mục gốc:", ROOT)

In [ ]:
#@title Ghi tệp dữ liệu thô nhúng sẵn vào data/raw (bấm để xem mã)
import base64, hashlib, zlib

EMBEDDED_FILES = {
    "data/raw/Students_Performance_data_set.xlsx": {
        "sha256": "04ebaf9f3a047afe96d9bff22dbe65b2a320e6395f230032735495e7e58277cf",
        "data": (
            "..."
        ),
    },
}

for rel, item in EMBEDDED_FILES.items():
    target = ROOT / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    raw = zlib.decompress(base64.b64decode(item["data"]))
    digest = hashlib.sha256(raw).hexdigest()
    assert digest == item["sha256"], f"Tệp {rel} bị hỏng khi giải nén"
    if target.exists():
        target.chmod(0o644)
    target.write_bytes(raw)
    print(f"{rel}: {len(raw):,} byte".replace(",", ".") + f", SHA-256 {digest[:16]}... khớp bản gốc")

In [ ]:
EXPECTED_SHA256 = "04ebaf9f3a047afe96d9bff22dbe65b2a320e6395f230032735495e7e58277cf"
EXPECTED_SHAPE = (1194, 31)
RANDOM_STATE = 42
N_FOLDS = 5

THRESHOLDS = {"y_lt_2_50": 2.50, "y_lt_2_75": 2.75, "y_lt_3_00": 3.00}
PRIMARY_TARGET = "y_lt_2_75"

# Cặp giá trị lặp bất thường (140 dòng) phát hiện ở bước profiling
PLACEHOLDER_SGPA = 2.68
PLACEHOLDER_CGPA = 3.15

# Ngưỡng miền (không học từ dữ liệu)
MAX_STANDARD_SEMESTER = 12      # chương trình BCSE chuẩn gồm 12 học kỳ
MAX_DAILY_HOURS = 16            # giới hạn giờ thức tối đa mỗi ngày cho một hoạt động
TIME_BUDGET_LIMIT = 18          # tổng giờ học + mạng xã hội + kỹ năng tối đa hợp lý
LOW_ATTENDANCE_LIMIT = 50       # chuyên cần dưới 50% được gắn cờ để rà soát

RAW_COLUMNS = [
    "University Admission year",
    "Gender",
    "Age",
    "H.S.C passing year",
    "Program",
    "Current Semester",
    "Do you have meritorious scholarship ?",
    "Do you use University transportation?",
    "How many hour do you study daily?",
    "How many times do you seat for study in a day?",
    "What is your preferable learning mode?",
    "Do you use smart phone?",
    "Do you have personal Computer?",
    "How many hour do you spent daily in social media?",
    "Status of your English language proficiency",
    "Average attendance on class",
    "Did you ever fall in probation?",
    "Did you ever got suspension?",
    "Do you attend in teacher consultancy for any kind of academical problems?",
    "What are the skills do you have ?",
    "How many hour do you spent daily on your skill development?",
    "What is you interested area?",
    "What is your relationship status?",
    "Are you engaged with any co-curriculum activities?",
    "With whom you are living with?",
    "Do you have any health issues?",
    "What was your previous SGPA?",
    "Do you have any physical disabilities?",
    "What is your current CGPA?",
    "How many Credit did you have completed?",
    "What is your monthly family income?",
]

COLUMN_MAP = dict(zip(RAW_COLUMNS, [
    "admit_year", "gender", "age", "hsc_year", "program", "semester",
    "scholarship", "transport", "study_hours", "study_sessions", "learning_mode",
    "smartphone", "pc", "social_hours", "english", "attendance", "probation",
    "suspension", "consultancy", "skills", "skill_hours", "interest_area",
    "relationship", "cocurricular", "living_with", "health_issue", "prev_sgpa",
    "disability", "cgpa", "credits_completed", "family_income",
]))

# Danh sách biến theo vai trò (căn cứ Đề cương Mục 10.4, 10.5 và Milestone 2 Mục 6.2)
LEAKAGE_COLUMNS = ["prev_sgpa", "probation", "suspension", "scholarship"]
REVERSE_DIRECTION_COLUMNS = ["consultancy"]            # chỉ dùng chẩn đoán
SENSITIVE_DROP_COLUMNS = ["family_income", "health_issue", "disability", "relationship"]
AUDIT_ONLY_COLUMNS = ["gender", "living_with"]         # chỉ kiểm định công bằng

CORE_AQ2 = ["study_hours", "study_sessions", "attendance_pct", "social_hours", "skill_hours"]
CORE_AQ3 = ["learning_mode_online", "has_pc", "uses_transport", "english_basic", "english_advance",
            "semester_capped", "credits_completed", "cocurricular"]   # 7 mục khảo sát, 8 cột (tiếng Anh = 2 biến giả)
CORE_FEATURES = CORE_AQ2 + CORE_AQ3
DERIVED_CANDIDATES = ["credits_per_semester", "study_hours_per_session", "study_minus_social"]
OPTIONAL_NOISY = ["skills_group", "interest_group"]

In [ ]:
class TransformationLog:
    """Ghi lại từng bước xử lý để phục vụ truy vết nguồn gốc dữ liệu."""

    def __init__(self) -> None:
        self.rows: list[dict] = []

    def add(self, step: str, description: str, rows_in: int, rows_out: int,
            n_changed: int | None = None) -> None:
        self.rows.append({
            "step": step, "description": description, "rows_in": rows_in,
            "rows_out": rows_out, "n_values_changed": n_changed,
        })

    def to_frame(self) -> pd.DataFrame:
        return pd.DataFrame(self.rows)


def sha256_of(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def yes_no_to_int(s: pd.Series) -> pd.Series:
    """Quy đổi Yes/No (không phân biệt hoa thường) về 1/0; giá trị lạ thành NA."""
    m = s.astype("string").str.strip().str.lower().map({"yes": 1, "y": 1, "no": 0, "n": 0})
    return m.astype("Int64")

In [ ]:
log = TransformationLog()
raw_df = pd.read_excel(raw_path, sheet_name="Students_Performance_data_set")
print("Bảng thô dùng để đối chiếu nhật ký chất lượng:", raw_df.shape)

In [ ]:
def step1_load_and_validate(raw_path: Path, log: TransformationLog) -> pd.DataFrame:
    """Bước 1: nạp tệp thô ở chế độ đọc và kiểm tra giả định về tệp và schema."""
    digest = sha256_of(raw_path)
    if digest != EXPECTED_SHA256:
        raise ValueError(f"SHA-256 không khớp Milestone 2: {digest}")
    df = pd.read_excel(raw_path, sheet_name="Students_Performance_data_set")
    if df.shape != EXPECTED_SHAPE:
        raise ValueError(f"Kích thước {df.shape} khác kỳ vọng {EXPECTED_SHAPE}")
    if list(df.columns) != RAW_COLUMNS:
        raise ValueError("Tên cột không khớp schema đã ghi ở Milestone 2")
    log.add("S1_load", "Nạp tệp thô, đối chiếu SHA-256, kích thước 1194x31 và tên 31 cột",
            len(df), len(df))
    return df


df = step1_load_and_validate(raw_path, log)
print(df.shape)

In [ ]:
def step2_rename_and_rowid(df: pd.DataFrame, log: TransformationLog) -> pd.DataFrame:
    """Bước 2: đổi tên cột sang dạng snake_case và tạo mã dòng rid (không có khóa gốc)."""
    out = df.rename(columns=COLUMN_MAP).copy()
    out.insert(0, "rid", np.arange(1, len(out) + 1))
    log.add("S2_rename", "Đổi tên 31 cột, thêm rid = thứ tự dòng trong tệp thô (giả định mỗi dòng "
            "là một sinh viên vì không có khóa định danh)", len(df), len(out))
    return out


df = step2_rename_and_rowid(df, log)
display(df.head(3))

In [ ]:
def step3_standardize_types(df: pd.DataFrame, log: TransformationLog) -> pd.DataFrame:
    """Bước 3: chuẩn hóa kiểu dữ liệu và nhãn phân loại."""
    out = df.copy()

    # 3a. Chuyên cần: cột bị đọc thành object vì có ô dạng khoảng "94-98"
    raw_att = out["attendance"].astype("string").str.strip()
    is_range = raw_att.str.match(r"^\d+(\.\d+)?\s*-\s*\d+(\.\d+)?$", na=False)

    def parse_att(v: str) -> float:
        if v is pd.NA or v is None:
            return np.nan
        nums = [float(x) for x in re.findall(r"\d+(?:\.\d+)?", str(v))]
        return float(np.mean(nums)) if nums else np.nan

    out["attendance_pct"] = raw_att.map(parse_att).astype(float)
    out["attendance_range_flag"] = is_range.astype(int)
    log.add("S3a_attendance", "Quy đổi chuyên cần về số; ô dạng khoảng lấy trung điểm (94-98 thành 96) "
            "và gắn cờ attendance_range_flag", len(out), len(out), int(is_range.sum()))

    # 3b. Nhãn Yes/No về 0/1
    for src, dst in [("transport", "uses_transport"), ("smartphone", "has_smartphone"),
                     ("pc", "has_pc"), ("cocurricular", "cocurricular_bin"),
                     ("scholarship", "scholarship_bin"), ("probation", "probation_bin"),
                     ("suspension", "suspension_bin"), ("consultancy", "consultancy_bin")]:
        out[dst] = yes_no_to_int(out[src])
    out["cocurricular"] = out["cocurricular_bin"]
    out["learning_mode_online"] = (out["learning_mode"].str.strip().str.lower() == "online").astype(int)
    out["english_level"] = out["english"].str.strip().map(
        {"Basic": 0, "Intermediate": 1, "Advance": 2}).astype(int)
    # Quan hệ giữa tiếng Anh và nhãn có dạng chữ U (xem EDA, AQ3) nên không mã hóa thứ bậc cho mô hình.
    # Dùng hai biến giả với Intermediate làm nhóm tham chiếu.
    out["english_basic"] = (out["english_level"] == 0).astype(int)
    out["english_advance"] = (out["english_level"] == 2).astype(int)
    log.add("S3b_binary", "Mã hóa Yes/No thành 1/0; learning_mode_online (Online = 1); english_level "
            "(Basic 0, Intermediate 1, Advance 2) chỉ giữ để mô tả; tiếng Anh vào mô hình bằng hai biến giả "
            "english_basic và english_advance (tham chiếu Intermediate)", len(out), len(out))

    # 3c. Chuẩn hóa nhãn không nhất quán ở các biến nhạy cảm (chỉ phục vụ kiểm toán)
    health_before = out["health_issue"].astype("string")
    rel_before = out["relationship"].astype("string")
    h = health_before.str.strip().str.lower().replace({"n": "no"})
    out["health_issue"] = h.str.capitalize()
    out["relationship"] = out["relationship"].replace({"In a relationship": "Relationship"})
    n_cat = int((health_before != out["health_issue"]).sum() + (rel_before != out["relationship"]).sum())
    log.add("S3c_labels", "Chuẩn hóa health_issue (Yes/yes/no/N thành Yes/No) và gộp nhãn đồng nghĩa "
            "của relationship; các biến này không đi vào tập đặc trưng", len(out), len(out), n_cat)

    # 3d. Gom nhóm hai trường văn bản tự do theo từ khóa (nhiễu cao, chỉ ở vai trò tùy chọn)
    def group_skill(v) -> str:
        if pd.isna(v):
            return "Unknown"
        t = str(v).lower()
        if re.search(r"nothing|no skill|not yet|haven't|don't|not any|do not know|^i don|learner|basic", t):
            return "None"
        if "cyber" in t:
            return "Cyber security"
        if "network" in t:
            return "Networking"
        if re.search(r"web|frontend", t):
            return "Web development"
        if re.search(r"artificial|machine", t):
            return "AI and ML"
        if re.search(r"program|software|python|app dev|database", t):
            return "Programming and software"
        return "Other"

    def group_interest(v) -> str:
        if pd.isna(v):
            return "Unknown"
        t = str(v).lower()
        if re.search(r"data|machine|artificial", t):
            return "Data and AI"
        if re.search(r"cyber|syber|network", t):
            return "Network and security"
        if re.search(r"ui/ux", t):
            return "UI/UX"
        if "hardware" in t:
            return "Hardware"
        if re.search(r"software|program|web|game|full stack", t):
            return "Software and web"
        return "Other"

    out["skills_group"] = out["skills"].map(group_skill)
    out["interest_group"] = out["interest_area"].map(group_interest)
    log.add("S3d_text_groups", "Gom skills (55 giá trị) và interest_area (25 giá trị) về nhóm rút gọn "
            "bằng quy tắc từ khóa; giá trị khuyết của skills thành Unknown",
            len(out), len(out), int(out["skills"].isna().sum()))
    return out


df = step3_standardize_types(df, log)
display(df[["attendance", "attendance_pct", "attendance_range_flag"]].loc[df["attendance_range_flag"] == 1])
display(df["english_level"].value_counts().sort_index().rename("Số sinh viên theo english_level (0 Basic, 1 Intermediate, 2 Advance)").to_frame())
display(df["skills_group"].value_counts().to_frame("Số sinh viên theo nhóm kỹ năng"))

In [ ]:
def step4_flag_quality(df: pd.DataFrame, log: TransformationLog) -> pd.DataFrame:
    """Bước 4: gắn cờ chất lượng (không xóa dòng nào)."""
    out = df.copy()
    out["placeholder_flag"] = ((out["prev_sgpa"].round(2) == PLACEHOLDER_SGPA) &
                               (out["cgpa"].round(2) == PLACEHOLDER_CGPA)).astype(int)
    out["cgpa_zero_flag"] = (out["cgpa"] == 0).astype(int)
    out["sgpa_zero_flag"] = (out["prev_sgpa"] == 0).astype(int)
    out["extended_duration_flag"] = (out["semester"] > MAX_STANDARD_SEMESTER).astype(int)
    out["hsc_invalid_flag"] = (out["hsc_year"] > out["admit_year"]).astype(int)
    out["attendance_low_flag"] = (out["attendance_pct"] < LOW_ATTENDANCE_LIMIT).astype(int)
    out["social_over_cap_flag"] = (out["social_hours"] > MAX_DAILY_HOURS).astype(int)
    budget = out["study_hours"] + out["social_hours"] + out["skill_hours"]
    out["time_budget_flag"] = (budget > TIME_BUDGET_LIMIT).astype(int)
    out["cgpa_extreme_low_flag"] = ((out["cgpa"] > 0) & (out["cgpa"] < 1.0)).astype(int)
    log.add("S4_flags", "Gắn 9 cờ chất lượng (placeholder, cgpa_zero, sgpa_zero, extended_duration, "
            "hsc_invalid, attendance_low, social_over_cap, time_budget, cgpa_extreme_low); không xóa dòng nào",
            len(out), len(out))
    return out


df = step4_flag_quality(df, log)
flag_cols = [c for c in df.columns if c.endswith("_flag")]
display(df[flag_cols].sum().rename("Số dòng bị gắn cờ").to_frame())

In [ ]:
def step5_handle_missing_invalid(df: pd.DataFrame, log: TransformationLog) -> pd.DataFrame:
    """Bước 5 và 6: xử lý giá trị khuyết và giá trị bất hợp lệ theo bằng chứng."""
    out = df.copy()

    # CGPA = 0: 37/39 dòng là sinh viên học kỳ 1 chưa có tín chỉ nên là "chưa có điểm"
    n0 = int(out["cgpa_zero_flag"].sum())
    out["cgpa_clean"] = out["cgpa"].where(out["cgpa"] > 0, np.nan)
    log.add("S5a_cgpa_zero", "CGPA = 0 chuyển thành khuyết (NaN) vì bằng chứng cho thấy đó là chưa có "
            "điểm tích lũy chứ không phải điểm thật", len(out), len(out), n0)

    # SGPA = 0 trong khi CGPA > 0 là mâu thuẫn; prev_sgpa không dùng làm đặc trưng nhưng vẫn làm sạch
    m = (out["prev_sgpa"] == 0) & (out["cgpa"] > 0)
    out["prev_sgpa_clean"] = out["prev_sgpa"].where(~((out["prev_sgpa"] == 0)), np.nan)
    log.add("S5b_sgpa_zero", "prev_sgpa = 0 chuyển thành khuyết; biến này thuộc nhóm rò rỉ nên chỉ giữ "
            "trong bảng trung gian để đối chứng", len(out), len(out), int((out["prev_sgpa"] == 0).sum()))

    # Năm tốt nghiệp THPT 2028 lớn hơn năm nhập học: lỗi nhập liệu
    out["hsc_year_clean"] = out["hsc_year"].where(out["hsc_invalid_flag"] == 0, np.nan)
    log.add("S5c_hsc", "hsc_year lớn hơn admit_year (2028 so với 2020) chuyển thành khuyết",
            len(out), len(out), int(out["hsc_invalid_flag"].sum()))

    # Học kỳ > 12: sinh viên học kéo dài hợp lệ (xem EDA), giữ nguyên và chặn trên ở biến dùng cho mô hình
    out["semester_capped"] = out["semester"].clip(upper=MAX_STANDARD_SEMESTER)
    log.add("S5d_semester", "Giữ nguyên semester; tạo semester_capped = min(semester, 12) và cờ "
            "extended_duration_flag vì các dòng này có tín chỉ và năm nhập học nhất quán",
            len(out), len(out), int(out["extended_duration_flag"].sum()))

    # Giờ mạng xã hội vượt giới hạn giờ thức: chặn theo ngưỡng miền (không học từ dữ liệu)
    n_cap = int((out["social_hours"] > MAX_DAILY_HOURS).sum())
    out["social_hours"] = out["social_hours"].clip(upper=MAX_DAILY_HOURS)
    log.add("S5e_social_cap", f"Chặn social_hours tại {MAX_DAILY_HOURS} giờ/ngày (ngưỡng miền); giá trị "
            "gốc vẫn còn trong tệp raw", len(out), len(out), n_cap)

    # CGPA cực thấp (<1,0): hợp lệ về miền 0-4 và nhãn luôn là dương, nên giữ nguyên
    log.add("S5g_cgpa_extreme", "Giữ nguyên 2 dòng CGPA dưới 1,0 (cùng học kỳ 2); nhãn y=1 ở mọi ngưỡng nên "
            "không ảnh hưởng tỷ lệ ca dương; chỉ gắn cờ cgpa_extreme_low_flag", len(out), len(out),
            int(out["cgpa_extreme_low_flag"].sum()))

    # Chuyên cần thấp (<50%): giữ nguyên, chỉ gắn cờ; không có cơ sở để coi là lỗi
    log.add("S5f_attendance_low", "Giữ nguyên chuyên cần thấp (<50%); chỉ gắn cờ vì không đủ cơ sở xác "
            "minh là lỗi nhập liệu", len(out), len(out), int(out["attendance_low_flag"].sum()))

    out["skills"] = out["skills"].fillna("Unknown")
    return out


df = step5_handle_missing_invalid(df, log)
print("CGPA khuyết sau xử lý:", int(df["cgpa_clean"].isna().sum()))
print("Giá trị lớn nhất của social_hours sau khi chặn:", df["social_hours"].max())
print("Giá trị lớn nhất của semester_capped:", df["semester_capped"].max())

In [ ]:
def step8_derive_features(df: pd.DataFrame, log: TransformationLog) -> pd.DataFrame:
    """Bước 8: xây dựng đặc trưng dẫn xuất từ trường thực tế, ghi rõ công thức."""
    out = df.copy()
    out["credits_per_semester"] = np.where(
        (out["semester"] > 0) & (out["credits_completed"] > 0),
        out["credits_completed"] / out["semester"], np.nan)
    out["study_hours_per_session"] = np.where(
        out["study_sessions"] > 0, out["study_hours"] / out["study_sessions"], np.nan)
    out["study_minus_social"] = out["study_hours"] - out["social_hours"]
    out["cohort_band"] = pd.cut(out["admit_year"], bins=[0, 2018, 2019, 2020, 2021, 2100],
                                labels=["<=2018", "2019", "2020", "2021", ">=2022"]).astype(str)
    out["semester_band"] = pd.cut(out["semester"], bins=[0, 4, 8, 100],
                                  labels=["Học kỳ 1-4", "Học kỳ 5-8", "Học kỳ từ 9"]).astype(str)
    out["age_band"] = pd.cut(out["age"], bins=[0, 20, 22, 100],
                             labels=["<=20", "21-22", ">=23"]).astype(str)
    log.add("S8_derived", "Tạo credits_per_semester = credits_completed / semester; "
            "study_hours_per_session = study_hours / study_sessions; study_minus_social = "
            "study_hours - social_hours; cohort_band, semester_band, age_band để phân tầng",
            len(out), len(out))
    return out


df = step8_derive_features(df, log)
display(df[["credits_completed", "semester", "credits_per_semester", "study_hours", "study_sessions",
            "study_hours_per_session", "study_minus_social", "cohort_band", "semester_band"]].head())

In [ ]:
def step9_targets_and_folds(df: pd.DataFrame, log: TransformationLog) -> pd.DataFrame:
    """Bước 9: xây dựng nhãn nhị phân, mẫu phân tích và chia fold phân tầng."""
    out = df.copy()
    valid = out["cgpa_clean"].notna()
    for name, thr in THRESHOLDS.items():
        out[name] = pd.array(np.where(valid, (out["cgpa_clean"] < thr).astype(int), -1), dtype="Int64")
        out.loc[~valid, name] = pd.NA

    out["in_primary_sample"] = (valid & (out["placeholder_flag"] == 0)).astype(int)
    out["in_sensitivity_B"] = valid.astype(int)

    # Chia fold phân tầng theo nhãn chính; không tách hold-out (số ca dương quá nhỏ)
    for sample_col, fold_col in [("in_primary_sample", "cv_fold_primary"),
                                 ("in_sensitivity_B", "cv_fold_sensB")]:
        out[fold_col] = pd.array([pd.NA] * len(out), dtype="Int64")
        idx = out.index[out[sample_col] == 1]
        y = out.loc[idx, PRIMARY_TARGET].astype(int).to_numpy()
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        for k, (_, test_pos) in enumerate(skf.split(np.zeros(len(idx)), y)):
            out.loc[idx[test_pos], fold_col] = k
    log.add("S9_targets_folds", "Tạo 3 nhãn nhị phân theo ngưỡng CGPA 2.50, 2.75, 3.00; mẫu chính "
            "(CGPA hợp lệ và không thuộc cặp lặp) và mẫu độ nhạy B (giữ cặp lặp); chia 5 fold "
            "phân tầng theo nhãn 2.75 với random_state=42", len(out), int(out["in_primary_sample"].sum()))
    return out


df = step9_targets_and_folds(df, log)
primary_view = df[df["in_primary_sample"] == 1]
fold_check = primary_view.groupby("cv_fold_primary")[PRIMARY_TARGET].agg(
    n="size", positives=lambda s: int(s.astype(int).sum()))
print("Mẫu chính:", len(primary_view), "| mẫu độ nhạy B:", int(df["in_sensitivity_B"].sum()))
display(fold_check)

In [ ]:
# Lưu bảng mô hình và bảng kiểm toán đúng như ô A4 của Milestone 4, rồi đối chiếu mã băm với Prepared Dataset v1.0
primary = df[df["in_primary_sample"] == 1]
model_cols = ["rid"] + CORE_FEATURES + list(THRESHOLDS) + ["cv_fold_primary"]
primary[model_cols].to_csv(processed / "m3_analytic_primary_model.csv", index=False)
df[["rid", "gender", "living_with", "age_band", "in_primary_sample", "in_sensitivity_B", PRIMARY_TARGET]].to_csv(
    processed / "m3_audit_groups.csv", index=False)
PREPARED_SHA256 = "5ee373340359f46d9ef286c6ff029b05aa0e3e7eb4bdb06a15e5765affeb4503"
data_sha = sha256_of(processed / "m3_analytic_primary_model.csv")
print("SHA-256 Prepared Dataset:", data_sha[:16] + "...",
      "khớp Prepared Dataset v1.0" if data_sha == PREPARED_SHA256 else "khác mã băm v1.0 (thường do phiên bản pandas ghi số khác); nội dung được kiểm tra bằng AUC ở Bước 3")

## Bước 3. Xử lý: mô hình M1 theo đúng cấu hình đã đăng ký trước

Mô hình chính M1 là hồi quy logistic L2 có trọng số lớp, chuẩn hóa và chọn tham số chỉ trong từng fold, kiểm định chéo phân tầng 5 phần × 10 lần với seed 42. Cấu hình này nằm trong kế hoạch đăng ký trước (SHA-256 `6544eb8a…`, kiểm tra ở notebook Milestone 4). Điểm dùng để xếp hạng là **điểm ngoài fold** (mỗi sinh viên chỉ được chấm bởi mô hình chưa từng thấy sinh viên đó), trung bình qua 10 lần lặp. Mốc so sánh B1 chỉ dùng chuyên cần.

In [ ]:
import hashlib
import json
import platform
import time
import warnings
from datetime import datetime, timezone
from math import ceil
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from scipy import stats
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scikit-learn", sklearn.__version__)

In [ ]:
# Hằng số lấy từ pipeline chuẩn bị dữ liệu của Milestone 3 (Phần A của notebook này), sao chép lại cho rõ ràng
CORE_FEATURES = ["study_hours", "study_sessions", "attendance_pct", "social_hours", "skill_hours",
                 "learning_mode_online", "has_pc", "uses_transport", "english_basic", "english_advance",
                 "semester_capped", "credits_completed", "cocurricular"]
PRIMARY_TARGET = "y_lt_2_75"
RANDOM_STATE = 42

BASE = Path("/content") if Path("/content").exists() else Path.cwd()
ROOT = BASE / "N02_Demo"
OUT = ROOT / "reports" / "milestone4"
FIG = OUT / "figures"
TAB = OUT / "tables"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

N_SPLITS, N_REPEATS, N_BOOT = 5, 10, 2000
FLAG_RATES = (0.15, 0.30)
RATE_GRID = tuple(np.round(np.arange(0.05, 0.501, 0.05), 2))

# ---------------------------------------------------------------------------
# 1. Cấu hình biến và nhóm tín hiệu
# ---------------------------------------------------------------------------
CONT = ["study_hours", "study_sessions", "attendance_pct", "social_hours", "skill_hours", "semester_capped", "credits_completed"]
BIN = ["learning_mode_online", "has_pc", "uses_transport", "english_basic", "english_advance", "cocurricular"]
GROUPS = {
    "Thói quen học tập": ["study_hours", "study_sessions", "attendance_pct", "social_hours", "skill_hours"],
    "Trình độ tiếng Anh": ["english_basic", "english_advance"],
    "Tiến độ học vụ": ["semester_capped", "credits_completed"],
    "Bối cảnh học tập": ["learning_mode_online", "has_pc", "uses_transport", "cocurricular"],
}
# 12 mục khảo sát: tiếng Anh là một mục nhưng có hai biến giả
ITEM_OF = {c: c for c in CORE_FEATURES}
ITEM_OF.update({"english_basic": "english", "english_advance": "english"})
N_ITEMS = len(set(ITEM_OF.values()))
LABEL = {
    "study_hours": "Giờ học mỗi ngày", "study_sessions": "Số lần ngồi học", "attendance_pct": "Chuyên cần (%)",
    "social_hours": "Giờ mạng xã hội", "skill_hours": "Giờ phát triển kỹ năng", "learning_mode_online": "Ưa học trực tuyến",
    "has_pc": "Có máy tính cá nhân", "uses_transport": "Dùng xe đưa đón", "english_basic": "Tiếng Anh Basic",
    "english_advance": "Tiếng Anh Advance", "semester_capped": "Học kỳ hiện tại", "credits_completed": "Tín chỉ hoàn thành",
    "cocurricular": "Hoạt động ngoại khóa", "english": "Trình độ tiếng Anh",
}
MODEL_NAME = {
    "B1": "B1 Baseline chuyên cần", "M1": "M1 Logistic (primary)", "M2": "M2 Random Forest",
    "M3": "M3 Gradient Boosting", "S1": "S1 Logistic L1 (chọn trong fold)",
}

In [ ]:
def lr_penalty(l1_ratio: float) -> dict:
    """Tham số phạt tương thích nhiều phiên bản scikit-learn (từ 1.8 dùng l1_ratio, bản cũ dùng penalty)."""
    major_minor = tuple(int(x) for x in sklearn.__version__.split(".")[:2])
    if major_minor >= (1, 8):
        return {"l1_ratio": l1_ratio}
    return {"penalty": "l1" if l1_ratio >= 1.0 else "l2"}


def vi(x: float, nd: int = 2) -> str:
    return f"{x:.{nd}f}".replace(".", ",")

In [ ]:
def auc_rank(y: np.ndarray, s: np.ndarray) -> float:
    """AUC theo thống kê xếp hạng Mann-Whitney (đồng hạng nhận hạng trung bình)."""
    r = stats.rankdata(s)
    n1 = y.sum()
    n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def hits_at_rate(y: np.ndarray, s: np.ndarray, rate: float) -> tuple[float, int]:
    """Số ca dương kỳ vọng nằm trong top rate*n điểm cao nhất; đồng hạng ở ngưỡng được chia đều."""
    n = len(y)
    k = ceil(rate * n)
    order = np.sort(s)[::-1]
    t = order[k - 1]
    above = s > t
    tie = s == t
    a = int(above.sum())
    b = int(tie.sum())
    pos = float(y[above].sum())
    if b > 0:
        pos += (k - a) / b * float(y[tie].sum())
    return pos, k


def recall_at(y, s, rate):
    pos, _ = hits_at_rate(y, s, rate)
    return pos / y.sum()


def precision_at(y, s, rate):
    pos, k = hits_at_rate(y, s, rate)
    return pos / k


def wilson(k: float, n: int, z: float = 1.96) -> tuple[float, float]:
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h


def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


def calib_slope_intercept(y, p):
    """Độ dốc và hệ số chặn hiệu chuẩn: hồi quy logistic của nhãn theo logit của xác suất dự đoán."""
    z = logit(p).reshape(-1, 1)
    lr = LogisticRegression(C=1e6, max_iter=1000)
    lr.fit(z, y)
    return float(lr.coef_[0, 0]), float(lr.intercept_[0])

In [ ]:
def preprocessor(cols: list[str]) -> ColumnTransformer:
    cont = [c for c in CONT if c in cols]
    binary = [c for c in BIN if c in cols]
    parts = []
    if cont:
        parts.append(("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", RobustScaler())]), cont))
    if binary:
        parts.append(("bin", Pipeline([("imp", SimpleImputer(strategy="most_frequent"))]), binary))
    return ColumnTransformer(parts)


def out_columns(cols: list[str]) -> list[str]:
    """Thứ tự cột đầu ra của preprocessor (liên tục trước, nhị phân sau)."""
    return [c for c in CONT if c in cols] + [c for c in BIN if c in cols]


C_GRID = {"C": list(np.logspace(-3, 1, 9))}


def inner_cv() -> StratifiedKFold:
    return StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE)


def make_model(key: str, cols: list[str], seed: int) -> Pipeline:
    pre = preprocessor(cols)
    if key in ("B1", "M1", "S2"):
        base = LogisticRegression(**lr_penalty(0.0), class_weight="balanced", max_iter=2000)
        est = GridSearchCV(base, C_GRID, scoring="roc_auc", cv=inner_cv(), n_jobs=1)
        return Pipeline([("pre", pre), ("clf", est)])
    if key == "M1u":
        base = LogisticRegression(**lr_penalty(0.0), max_iter=2000)
        est = GridSearchCV(base, C_GRID, scoring="roc_auc", cv=inner_cv(), n_jobs=1)
        return Pipeline([("pre", pre), ("clf", est)])
    if key == "M2":
        return Pipeline([("pre", pre), ("clf", RandomForestClassifier(
            n_estimators=300, min_samples_leaf=10, max_features="sqrt", class_weight="balanced_subsample",
            n_jobs=2, random_state=seed))])
    if key == "M3":
        return Pipeline([("pre", pre), ("clf", HistGradientBoostingClassifier(
            max_depth=3, learning_rate=0.05, max_iter=200, l2_regularization=1.0, class_weight="balanced",
            early_stopping=False, random_state=seed))])
    if key == "S1":
        return Pipeline([("pre", pre), ("sel", L1Select()), ("clf", LogisticRegression(
            **lr_penalty(0.0), class_weight="balanced", max_iter=2000))])
    raise ValueError(key)


class L1Select(BaseEstimator, TransformerMixin):
    """Chọn đặc trưng bằng hồi quy logistic L1; chọn C bằng CV lồng 3 phần theo AUC; giữ ít nhất một đặc trưng."""

    def fit(self, X, y):
        l1 = LogisticRegression(**lr_penalty(1.0), solver="saga", class_weight="balanced", max_iter=5000, tol=1e-3,
                                random_state=RANDOM_STATE)
        grid = GridSearchCV(l1, C_GRID, scoring="roc_auc", cv=inner_cv(), n_jobs=1)
        grid.fit(X, y)
        coef = np.abs(grid.best_estimator_.coef_).ravel()
        mask = coef > 1e-8
        if not mask.any():
            mask[int(np.argmax(coef))] = True
        self.mask_ = mask
        self.best_C_ = float(grid.best_params_["C"])
        return self

    def get_support(self):
        return self.mask_

    def transform(self, X):
        return X[:, self.mask_]

In [ ]:
def split_list(X: pd.DataFrame, y: np.ndarray):
    rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
    return list(rskf.split(X, y))


def run_cv(key: str, X: pd.DataFrame, y: np.ndarray, splits, cols: list[str] | None = None, record: str | None = None):
    """Chạy CV lặp; trả về ma trận điểm ngoài fold (N_REPEATS x n) và thông tin từng lần fit."""
    cols = cols or list(X.columns)
    n = len(y)
    S = np.zeros((N_REPEATS, n))
    P = np.zeros((N_REPEATS, n))   # xác suất hiệu chỉnh theo tỷ lệ nền huấn luyện (dùng cho mô hình logistic)
    info = []
    for i, (tr, te) in enumerate(splits):
        rep = i // N_SPLITS
        pipe = make_model(key, cols, seed=RANDOM_STATE + rep)
        pipe.fit(X.iloc[tr][cols], y[tr])
        p = pipe.predict_proba(X.iloc[te][cols])[:, 1]
        S[rep, te] = p
        if key in ("M1", "B1", "S2"):
            pi = y[tr].mean()
            P[rep, te] = 1 / (1 + np.exp(-(logit(p) + np.log(pi / (1 - pi)))))
        else:
            P[rep, te] = p
        rec = {"rep": rep, "fold": i % N_SPLITS}
        if record == "lr":
            clf = pipe.named_steps["clf"]
            rec["coef"] = dict(zip(out_columns(cols), clf.best_estimator_.coef_.ravel()))
            rec["C"] = float(clf.best_params_["C"])
        if record == "l1":
            mask = pipe.named_steps["sel"].get_support()
            names = np.array(out_columns(cols))[mask]
            rec["selected_cols"] = list(names)
            rec["selected_items"] = sorted({ITEM_OF[c] for c in names})
            rec["C"] = pipe.named_steps["sel"].best_C_
        if record == "perm":
            rng = np.random.default_rng(RANDOM_STATE + i)
            Xte = X.iloc[te][cols].copy()
            base_auc = auc_rank(y[te], pipe.predict_proba(Xte)[:, 1])
            drops = {}
            for g, gc in GROUPS.items():
                gc = [c for c in gc if c in cols]
                if not gc:
                    continue
                d = []
                for _ in range(10):
                    Xp = Xte.copy()
                    idx = rng.permutation(len(Xp))
                    Xp[gc] = Xp[gc].to_numpy()[idx]
                    d.append(base_auc - auc_rank(y[te], pipe.predict_proba(Xp)[:, 1]))
                drops[g] = float(np.mean(d))
            rec["perm_drop"] = drops
        info.append(rec)
    return S, P, info

In [ ]:
t0 = time.time()
data = pd.read_csv(ROOT / "data" / "processed" / "m3_analytic_primary_model.csv")
audit = pd.read_csv(ROOT / "data" / "processed" / "m3_audit_groups.csv")[["rid", "gender", "living_with"]]
assert len(data) == 1015 and data[CORE_FEATURES].isna().sum().sum() == 0
assert not {"gender", "living_with", "age", "prev_sgpa", "probation", "suspension", "scholarship"} & set(CORE_FEATURES)
X = data[CORE_FEATURES]
y = data[PRIMARY_TARGET].to_numpy().astype(int)
n, npos = len(y), int(y.sum())
splits = split_list(X, y)
S_M1, P_M1, info_M1 = run_cv("M1", X, y, splits, record="lr")
S_B1, _, _ = run_cv("B1", X, y, splits, cols=["attendance_pct"])
auc_m1 = np.mean([auc_rank(y, S_M1[r]) for r in range(N_REPEATS)])
auc_b1 = np.mean([auc_rank(y, S_B1[r]) for r in range(N_REPEATS)])
score = S_M1.mean(axis=0)   # điểm ngoài fold trung bình, dùng để xếp hạng
print(f"n = {n}, số sinh viên cần hỗ trợ (CGPA < 2,75) = {npos} ({npos / n:.1%})")
print(f"AUC M1 = {auc_m1:.3f} (báo cáo: 0,621) | AUC chỉ chuyên cần B1 = {auc_b1:.3f} (báo cáo: 0,604) | {time.time() - t0:.0f} giây")
print("Khớp báo cáo trong sai lệch 0,005:", abs(auc_m1 - 0.621) <= 0.005 and abs(auc_b1 - 0.604) <= 0.005)

## Bước 4. Đầu ra: danh sách ưu tiên kèm reason code

**Cách tính reason code.** Mô hình M1 được huấn luyện lại trên toàn bộ mẫu với cùng cấu hình. Với mỗi sinh viên, đóng góp của một thông tin bằng hệ số chuẩn hóa nhân với độ lệch của sinh viên đó so với trung bình (sau chuẩn hóa). Học kỳ và số tín chỉ được gộp thành một nhóm "tiến độ học vụ" vì hai biến này đi rất sát nhau. Mỗi sinh viên được hiển thị hai đến ba lý do làm tăng thứ hạng nhiều nhất, viết bằng ngôn ngữ hỗ trợ.

Lý do chỉ giải thích **vì sao công cụ xếp sinh viên lên trước**, không phải nguyên nhân của kết quả học tập.

In [ ]:
final = make_model("M1", CORE_FEATURES, seed=RANDOM_STATE).fit(X, y)
Z = final.named_steps["pre"].transform(X)
cols_out = out_columns(CORE_FEATURES)
coef = final.named_steps["clf"].best_estimator_.coef_.ravel()
contrib = pd.DataFrame((Z - Z.mean(axis=0)) * coef, columns=cols_out, index=data.index)
contrib["tien_do"] = contrib.pop("semester_capped") + contrib.pop("credits_completed")
display(pd.Series(dict(zip(cols_out, coef))).rename("Hệ số chuẩn hóa của M1 (huấn luyện trên toàn mẫu)").round(3).to_frame())


def phrase(key: str, row: pd.Series) -> str:
    """Câu lý do bằng ngôn ngữ hỗ trợ; không dùng nhãn tiêu cực."""
    v = row
    return {
        "attendance_pct": f"Chuyên cần tự khai {v.attendance_pct:.0f}%: nên hỏi về khó khăn khi đến lớp",
        "english_basic": "Tự đánh giá tiếng Anh mức Basic: có thể giới thiệu lớp hỗ trợ tiếng Anh",
        "english_advance": "Tiếng Anh mức Advance: nhóm này cũng cần hỗ trợ nhiều hơn mức trung bình, nên hỏi thêm về việc học",
        "tien_do": f"Tiến độ học vụ (học kỳ {int(v.semester_capped)}, {int(v.credits_completed)} tín chỉ): nên cùng rà soát kế hoạch học tập",
        "study_hours": f"Giờ tự học mỗi ngày {v.study_hours:.0f} giờ: có thể trao đổi về cách sắp xếp thời gian học",
        "study_sessions": f"Số lần ngồi học mỗi ngày {v.study_sessions:.0f}: có thể trao đổi về nhịp học",
        "social_hours": f"Giờ mạng xã hội mỗi ngày {v.social_hours:.0f} giờ: có thể trao đổi về cân bằng thời gian",
        "skill_hours": f"Giờ phát triển kỹ năng mỗi ngày {v.skill_hours:.0f} giờ: có thể trao đổi về cân bằng thời gian",
        "learning_mode_online": "Ưa học trực tuyến: nên hỏi về khả năng tự quản lý việc học",
        "has_pc": "Không có máy tính cá nhân: có thể giới thiệu hỗ trợ thiết bị",
        "uses_transport": "Không dùng xe đưa đón của trường: có thể hỏi về việc đi lại",
        "cocurricular": "Tham gia hoạt động ngoại khóa: có thể trao đổi về cân bằng thời gian",
    }[key]


def reasons(i: int, k_min: int = 2, k_max: int = 3) -> list[str]:
    c = contrib.loc[i].sort_values(ascending=False)
    keys = [k for k, val in c.items() if val > 0][:k_max]   # chỉ lý do làm tăng thứ hạng
    return [phrase(k, data.loc[i]) for k in keys] or ["Không có lý do nổi bật; cố vấn xem xét trực tiếp"]


k = ceil(CAPACITY)
order = np.argsort(-score, kind="stable")
top = order[:k]
pct = pd.Series(score).rank(pct=True)
rows = []
for rank, i in enumerate(top, 1):
    rs = reasons(i) + ["", ""]
    rows.append({"Hạng": rank,
                 "Mã giả danh": "SV-" + hashlib.sha256(f"N02-demo-{int(data.loc[i, 'rid'])}".encode()).hexdigest()[:6].upper(),
                 "Mức ưu tiên (bách phân vị)": int(round(100 * pct[i])),
                 "Lý do 1": rs[0], "Lý do 2": rs[1], "Lý do 3": rs[2], "Trạng thái": "Chờ duyệt", "_idx": int(i)})
queue = pd.DataFrame(rows)
print(f"Danh sách ưu tiên: {k} sinh viên ({k / n:.1%} khóa). Không có cột giới tính, nơi ở hay điểm số dạng xác suất.")
display(queue.drop(columns="_idx").head(15))

In [ ]:
# Lý do nào xuất hiện nhiều nhất trong danh sách (giúp cố vấn chuẩn bị nội dung trao đổi)
first_words = queue["Lý do 1"].str.split(":").str[0].str.replace(r"\d+", "…", regex=True)
display(first_words.value_counts().rename("Số sinh viên có lý do này ở vị trí 1").to_frame())

## Bước 5. Bằng chứng: công cụ đáng tin đến đâu ở mức danh sách này

Các chỉ số dưới đây tính trên điểm ngoài fold và nhãn thật của dữ liệu lịch sử, theo đúng cách của Milestone 4 (trung bình qua 10 lần lặp). Công bằng được báo ở dạng số gộp theo nhóm; danh sách ở Bước 4 không chứa thuộc tính nhạy cảm.

In [ ]:
rate = k / n
rec = np.mean([recall_at(y, S_M1[r], rate) for r in range(N_REPEATS)])
prec = np.mean([precision_at(y, S_M1[r], rate) for r in range(N_REPEATS)])
rec_b1 = np.mean([recall_at(y, S_B1[r], rate) for r in range(N_REPEATS)])
print(f"Gắn cờ {k} sinh viên ({rate:.1%}):")
print(f"  Precision ≈ {prec:.1%}: khoảng {round(prec * 100)} trong 100 sinh viên được gọi thật sự cần hỗ trợ (chọn ngẫu nhiên: {npos / n:.1%})")
print(f"  Recall ≈ {rec:.1%}: tìm thấy khoảng {round(rec * npos)} trên {npos} sinh viên cần hỗ trợ; còn khoảng {npos - round(rec * npos)} bạn bị bỏ sót")
print(f"  Chỉ nhìn chuyên cần (B1) ở cùng mức: recall ≈ {rec_b1:.1%}")
print(f"  AUC M1 {auc_m1:.3f} so với B1 {auc_b1:.3f}: công cụ chỉ nhỉnh hơn việc nhìn chuyên cần; không dùng thay phán đoán của cố vấn.")

aud = data[["rid"]].merge(audit, on="rid", how="left")
thr = np.sort(score)[::-1][k - 1]
flag = score >= thr
fair = []
for attr, label in (("gender", "Giới tính"), ("living_with", "Nơi ở")):
    for g in sorted(aud[attr].dropna().unique()):
        m = (aud[attr] == g).to_numpy()
        pos = int(y[m].sum()); hit = int((flag & m & (y == 1)).sum())
        lo, hi = wilson(hit, pos)
        fair.append({"Thuộc tính": label, "Nhóm": g, "Số ca cần hỗ trợ": pos, "Recall": round(hit / pos, 3),
                     "KTC 95%": f"[{lo:.2f}; {hi:.2f}]"})
fair = pd.DataFrame(fair)
display(fair)
print("Khoảng tin cậy rộng và chồng lên nhau: chưa kết luận được công cụ công bằng hay thiên lệch; cần theo dõi mỗi học kỳ (cổng G3).")

## Bước 6. Con người phê duyệt từng ca (cổng G2) và nhật ký kiểm toán

Cố vấn chọn một trong ba quyết định cho từng sinh viên: **Duyệt liên hệ**, **Bỏ qua** (bắt buộc ghi lý do) hoặc **Chuyển giảng viên**. Mỗi thao tác được ghi vào nhật ký kiểm toán cùng phiên bản mô hình và mã băm dữ liệu. Không có thông báo nào được gửi tự động.

In [ ]:
PLAN_SHA256 = "6544eb8a5875a0ce8db2733720b9b4300c685cc9d1c9afe8a9f4918915681215"
ALLOWED = {"Duyệt liên hệ", "Bỏ qua", "Chuyển giảng viên"}
audit_log = []


def review(pseudo: str, decision: str, reason: str = "", reviewer: str = ADVISOR) -> None:
    """Cổng G2: ghi quyết định của cố vấn cho một sinh viên."""
    if decision not in ALLOWED:
        raise ValueError(f"Quyết định phải là một trong {sorted(ALLOWED)}")
    if decision == "Bỏ qua" and not reason.strip():
        raise ValueError("Bỏ qua phải ghi lý do")
    m = queue["Mã giả danh"] == pseudo
    if not m.any():
        raise KeyError(pseudo)
    queue.loc[m, "Trạng thái"] = decision
    audit_log.append({"thoi_gian_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"), "nguoi_duyet": reviewer,
                      "ma_gia_danh": pseudo, "quyet_dinh": decision, "ly_do": reason,
                      "mo_hinh": "M1 logistic L2 (điểm ngoài fold)", "sha256_ke_hoach": PLAN_SHA256[:12],
                      "sha256_du_lieu": data_sha[:12]})


# MÔ PHỎNG cho buổi demo: cố vấn xem năm ca đầu. Khi dùng thật, cố vấn tự nhập quyết định của mình.
p = queue["Mã giả danh"].tolist()
review(p[0], "Duyệt liên hệ", "Hẹn gặp trong tuần")
review(p[1], "Duyệt liên hệ", "Gửi lời mời trao đổi")
review(p[2], "Bỏ qua", "Sinh viên đang được giảng viên kèm riêng")
review(p[3], "Chuyển giảng viên", "Cần hỗ trợ nội dung môn học")
review(p[4], "Duyệt liên hệ", "Hẹn gặp trong tuần")
display(queue.drop(columns="_idx").head(6))
display(pd.DataFrame(audit_log))

## Bước 7. Hành động: xuất danh sách đã duyệt

Danh sách chỉ được xuất khi **mọi ca đã có quyết định của cố vấn**. Ô dưới cố xuất ngay khi còn ca chưa duyệt để minh họa cổng chặn; sau đó cố vấn chủ động khép phiên (các ca còn lại ghi Bỏ qua kèm lý do), rồi mới xuất được.

In [ ]:
def export_approved() -> pd.DataFrame:
    pending = int((queue["Trạng thái"] == "Chờ duyệt").sum())
    if pending:
        raise PermissionError(f"Từ chối xuất: còn {pending} ca chưa có quyết định của cố vấn (cổng G2).")
    out = queue[queue["Trạng thái"].isin(["Duyệt liên hệ", "Chuyển giảng viên"])].drop(columns="_idx")
    out_dir = ROOT / "demo_output"
    out_dir.mkdir(parents=True, exist_ok=True)
    out.to_csv(out_dir / "danh_sach_da_duyet.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(audit_log).to_csv(out_dir / "nhat_ky_kiem_toan.csv", index=False, encoding="utf-8-sig")
    return out


try:
    export_approved()
except PermissionError as e:
    print(e)

In [ ]:
def close_session(reason: str, reviewer: str = ADVISOR) -> None:
    """Cố vấn khép phiên: các ca chưa xem được ghi Bỏ qua kèm lý do (vẫn ghi nhật ký từng ca)."""
    for pseudo in queue.loc[queue["Trạng thái"] == "Chờ duyệt", "Mã giả danh"]:
        review(pseudo, "Bỏ qua", reason, reviewer)


close_session("Hết năng lực tuần này; xem lại ở phiên tuần sau")
approved = export_approved()
print(f"Đã xuất {len(approved)} ca đã duyệt và nhật ký {len(audit_log)} dòng vào {ROOT / 'demo_output'}")
display(approved)

print("""
Mẫu lời mời (ngôn ngữ hỗ trợ, không nhắc đến điểm số hay "nguy cơ"):
  Chào bạn, phòng cố vấn học tập muốn mời bạn trò chuyện khoảng 15 phút về việc học trong học kỳ này,
  để xem trường có thể hỗ trợ gì thêm cho bạn. Bạn chọn giúp một khung giờ phù hợp nhé.
""")

## Tóm tắt demo và giới hạn

| Bước | Điều đã thấy trong demo |
|---|---|
| Câu hỏi | Cố vấn chỉ gọi được khoảng 15% khóa (153 sinh viên) |
| Đầu vào | Tệp gốc đúng SHA-256; Prepared Dataset v1.0 (1.015 sinh viên) dựng lại từ mã Milestone 3 |
| Xử lý | M1 với cấu hình đã đăng ký trước; AUC khớp báo cáo |
| Đầu ra | Danh sách mã giả danh kèm 2–3 reason code bằng ngôn ngữ hỗ trợ |
| Bằng chứng | Precision, recall, so với chỉ nhìn chuyên cần; công bằng ở dạng số gộp, chưa kết luận được |
| Phê duyệt | Mỗi ca có quyết định của cố vấn; Bỏ qua phải ghi lý do; mọi thao tác vào nhật ký |
| Hành động | Xuất bị chặn khi còn ca chưa duyệt; chỉ xuất ca đã duyệt cùng nhật ký kiểm toán |

**Giới hạn:** dữ liệu của một trường ở Bangladesh, tự khai và cắt ngang; công cụ chỉ là lớp xếp ưu tiên bổ trợ, bỏ sót hơn một nửa sinh viên cần hỗ trợ ở mức 15%; reason code chỉ giải thích thứ hạng, không phải nguyên nhân; quyết định duyệt trong demo là mô phỏng.